# LSTM Training-Window Robustness

Training-length robustness check for the LSTM. The counterpart to tft_training_window_robustness.ipynb, and part of the same robustness section as the main experiment in validation_regime_experiment.ipynb.

The question: does the calm-versus-stress gap survive when the training-window length is varied, holding the validation windows (calm 2013-14, stress 2015-16) and the 2020+ test fixed?

Stage 1 sets up the data and the QLIKE-selection harness. Stage 2 is the LSTM floor check. It varies the training size across five lengths with three seeds each to find the shortest length where the LSTM stays stable, which is how the shortest sweep arm was chosen. Stage 3 is the sweep. For each training start (2000_full, 2004_med, 2005_short) it runs a fresh Optuna search on the calm window and another on the stress window, then reports the high-volatility gap at the 33/67 and 25/75 thresholds with a Diebold-Mariano test.

The LSTM uses QLIKE selection throughout, matching the main experiment. The second cell runs four self-checks on the harness. Data is pulled from Yahoo Finance at run time.

In [ ]:
# ============================================================================
# OPTION 2 — TRAINING-LENGTH ROBUSTNESS OF THE VALIDATION-REGIME EFFECT
# Complete, self-contained: data fetch -> LSTM floor check -> sweep (both thresholds)
# ----------------------------------------------------------------------------
# Tests whether Paper 1's calm-vs-stress high-vol QLIKE gap survives when the
# TRAINING-window length is varied, holding validation (calm 2013-14 / stress
# 2015-16) and the 2020+ test FIXED. LSTM-led (TFT floor ~3,860 forbids short arms).
#
# STAGE 1: setup (yfinance, Garman-Klass, windows, QLIKE-objective harness)
# STAGE 2: LSTM floor check — see where the LSTM destabilises BEFORE picking the
#          shortest training arm (so we don't go so low it hurts LSTM performance)
# STAGE 3: the sweep — both arms x training lengths x BOTH thresholds (33/67, 25/75)
#
# Run top-to-bottom in one cell. Read STAGE 2 output before trusting STAGE 3:
# pick the shortest arm at/above the LSTM floor.
# ============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightning.pytorch as pl

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================================
# STAGE 1 — DATA FETCH + SETUP (verbatim from the frozen Paper 1 pipeline)
# ============================================================================
import yfinance as yf
sp500 = yf.download("^GSPC", start="2000-08-01", end="2025-01-22",
                    progress=False, auto_adjust=False)
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)
sp500 = sp500[["Open", "High", "Low", "Close"]].copy()
sp500["returns"] = sp500["Close"].pct_change()
sp500["squared_returns"] = sp500["returns"] ** 2
sp500["rv_gk"] = (0.5 * (np.log(sp500["High"] / sp500["Low"])) ** 2
                  - (2 * np.log(2) - 1) * (np.log(sp500["Close"] / sp500["Open"])) ** 2)
sp500 = sp500.dropna()
sp500_dt = sp500.copy()
print(f"Total obs: {len(sp500)} ({sp500.index.min().date()} -> {sp500.index.max().date()})")

# ---- Window boundaries (Paper 1) ----
TRAIN_END   = "2013-01-01"
CALM_START, CALM_END     = "2013-01-01", "2015-01-01"
STRESS_START, STRESS_END = "2015-01-01", "2017-01-01"
TEST_START  = "2020-01-01"

def ann_vol(mask):
    d = sp500_dt.loc[mask, "rv_gk"]
    return np.sqrt(d.mean() * 252) * 100
print(f"Calm  2013-14 vol: {ann_vol((sp500_dt.index>=CALM_START)&(sp500_dt.index<CALM_END)):.2f}%")
print(f"Stress2015-16 vol: {ann_vol((sp500_dt.index>=STRESS_START)&(sp500_dt.index<STRESS_END)):.2f}%")

# ---- Shared helpers (verbatim) ----
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i - seq_length:i]); y.append(data[i])
    return np.array(X), np.array(y)

def calculate_qlike(actual, predicted):
    actual = np.array(actual).flatten(); predicted = np.array(predicted).flatten()
    eps = 1e-10
    predicted = np.maximum(predicted, eps); actual = np.maximum(actual, eps)
    return np.mean(actual / predicted - np.log(actual / predicted) - 1)

class LSTMVolatility(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# ---- data slicing with a TRAIN_START (only addition vs Paper 1's lstm_data) ----
def lstm_data_ts(train_start, val_start, val_end):
    train = sp500_dt[(sp500_dt.index >= train_start) & (sp500_dt.index < TRAIN_END)]
    val   = sp500_dt[(sp500_dt.index >= val_start) & (sp500_dt.index < val_end)]
    test  = sp500_dt[sp500_dt.index >= TEST_START]
    tr = train["rv_gk"].values.reshape(-1, 1)
    va = val["rv_gk"].values.reshape(-1, 1)
    te = test["rv_gk"].values.reshape(-1, 1)
    scaler = StandardScaler()
    tr_s = scaler.fit_transform(tr)      # fit on TRAIN only (leakage audit pt 4)
    va_s = scaler.transform(va); te_s = scaler.transform(te)
    return tr_s, va_s, te_s, scaler, len(train)

# ---- train/eval (QLIKE selection), parameterised by train_start ----
def lstm_train_eval_ts(train_start, seq_length, hidden, layers, dropout, lr, batch,
                       val_start, val_end, epochs=100, patience=10, seed=SEED):
    pl.seed_everything(seed, workers=True)
    tr_s, va_s, te_s, scaler, n_train = lstm_data_ts(train_start, val_start, val_end)
    va_h = np.vstack([tr_s[-seq_length:], va_s])
    te_h = np.vstack([va_s[-seq_length:], te_s])
    Xtr, ytr = create_sequences(tr_s, seq_length)
    Xva, yva = create_sequences(va_h, seq_length)
    Xte, yte = create_sequences(te_h, seq_length)
    Xtr, ytr = torch.FloatTensor(Xtr), torch.FloatTensor(ytr)
    Xva, yva = torch.FloatTensor(Xva), torch.FloatTensor(yva)
    Xte = torch.FloatTensor(Xte)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, shuffle=True)

    model = LSTMVolatility(1, hidden, layers, dropout)
    crit = nn.MSELoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, wait = float("inf"), None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vloss = crit(model(Xva), yva).item()
        if vloss < best:
            best, best_state, wait = vloss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()

    with torch.no_grad():
        val_pred = model(Xva).numpy()
    val_pred_orig   = scaler.inverse_transform(val_pred).flatten()
    val_actual_orig = scaler.inverse_transform(yva.numpy().reshape(-1, 1)).flatten()
    val_qlike = calculate_qlike(val_actual_orig, val_pred_orig)

    with torch.no_grad():
        pred = model(Xte).numpy()
    pred_orig = scaler.inverse_transform(pred).flatten()
    return pred_orig, best, val_qlike, n_train

def lstm_objective_factory_ts(train_start, val_start, val_end):
    def objective(trial):
        seq_length = trial.suggest_categorical("seq_length", [22, 44, 66])
        hidden     = trial.suggest_categorical("hidden_size", [64, 128])
        layers     = trial.suggest_int("num_layers", 1, 2)
        dropout    = trial.suggest_float("dropout", 0.1, 0.3, step=0.1)
        lr         = trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True)
        batch      = trial.suggest_categorical("batch_size", [32, 64])
        _, _, vq, _ = lstm_train_eval_ts(train_start, seq_length, hidden, layers,
                                         dropout, lr, batch, val_start, val_end,
                                         epochs=100, patience=10)
        return vq
    return objective

def run_lstm_arm_ts(train_start, val_start, val_end, label, n_trials=30):
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10))
    study.optimize(lstm_objective_factory_ts(train_start, val_start, val_end), n_trials=n_trials)
    bp = study.best_params
    pred_orig, _, vq, n_train = lstm_train_eval_ts(
        train_start, bp["seq_length"], bp["hidden_size"], bp["num_layers"],
        bp["dropout"], bp["learning_rate"], bp["batch_size"], val_start, val_end)
    print(f"  {label}: n_train={n_train} | val QLIKE={vq:.4f} | {bp}")
    return pred_orig, n_train

# ---- evaluation helpers (verbatim) ----
def qlike_vec(a, f):
    eps = 1e-10; a = np.maximum(a, eps); f = np.maximum(f, eps)
    return a/f - np.log(a/f) - 1

def dm_test(a, f_calm, f_stress, mask, h_step=1):
    d = qlike_vec(a, f_calm)[mask] - qlike_vec(a, f_stress)[mask]   # +ve => calm worse
    nm = len(d); dbar = d.mean()
    h = max(h_step-1, int(np.floor(4*(nm/100.0)**(2.0/9.0))))
    g0 = np.sum((d-dbar)**2)/nm; var = g0
    for k in range(1, h+1):
        ck = np.sum((d[k:]-dbar)*(d[:-k]-dbar))/nm
        var += 2*(1-k/(h+1))*ck
    dm = dbar/np.sqrt(var/nm)
    hln = dm*np.sqrt((nm+1-2*h_step+h_step*(h_step-1)/nm)/nm)
    return dbar, hln, nm

# ============================================================================
# STAGE 2 — LSTM FLOOR CHECK
# How low can the training window go before the LSTM itself destabilises?
# Mirrors the TFT floor-check logic: fixed sensible config, vary train size,
# 3 seeds, read val QLIKE + seed-std + pred/actual variance ratio BY EYE.
# Pick the shortest sweep arm at/above where this stays stable.
# ============================================================================
print("\n\n############ STAGE 2: LSTM FLOOR CHECK ############")
print("Vary training size; watch val QLIKE, seed-std, and pred_var_ratio.")
print("Floor = shortest size where val QLIKE stays low, seed-std stays small,")
print("and pred_var_ratio does NOT collapse toward 0. Read by eye.\n")

# Fixed, representative LSTM config for the floor probe (mid of search space)
FLOOR_CFG = dict(seq_length=22, hidden=64, layers=1, dropout=0.1, lr=1e-3, batch=32)
FLOOR_STARTS = {
    "2000 (~3122d)": "2000-08-01",
    "2004 (~2260d)": "2004-01-01",
    "2006 (~1760d)": "2006-01-01",
    "2008 (~1255d)": "2008-01-01",
    "2010 (~755d)":  "2010-01-01",
}
FLOOR_SEEDS = [0, 1, 2]
# Use the CALM validation window for the floor probe (selection target fixed)
floor_rows = []
for label, tstart in FLOOR_STARTS.items():
    qs, ratios, ntr = [], [], None
    for sd in FLOOR_SEEDS:
        pred, _, vq, n_train = lstm_train_eval_ts(
            tstart, FLOOR_CFG["seq_length"], FLOOR_CFG["hidden"], FLOOR_CFG["layers"],
            FLOOR_CFG["dropout"], FLOOR_CFG["lr"], FLOOR_CFG["batch"],
            CALM_START, CALM_END, epochs=100, patience=10, seed=sd)
        ntr = n_train
        # pred/actual variance ratio on the test forecasts (collapse diagnostic)
        test = sp500_dt[sp500_dt.index >= TEST_START]
        act = test["rv_gk"].values
        m = min(len(act), len(pred)); 
        pv = float(np.var(pred[-m:])); av = float(np.var(act[-m:]))
        qs.append(vq); ratios.append(pv/av if av>0 else np.nan)
    floor_rows.append(dict(label=label, n_train=ntr,
                           q_mean=np.mean(qs), q_std=np.std(qs),
                           ratio_mean=np.mean(ratios)))

print(f"{'train window':<16}{'n_train':>9}{'valQLIKE_mean':>15}{'valQLIKE_std':>14}{'pred_var_ratio':>16}")
for r in floor_rows:
    print(f"{r['label']:<16}{r['n_train']:>9}{r['q_mean']:>15.4f}{r['q_std']:>14.4f}{r['ratio_mean']:>16.3f}")
print("\n>>> READ THIS before Stage 3: the shortest training window where valQLIKE_std")
print(">>> is still small and pred_var_ratio is NOT collapsing is the LSTM floor.")
print(">>> Stage 3 below sweeps 2000/2004/2008 — if the floor check shows 2008 is")
print(">>> unstable, drop the 2008 arm and rerun Stage 3 with a higher shortest arm.\n")

# ============================================================================
# STAGE 3 — TRAINING-LENGTH SWEEP (short arm = 2005, both thresholds)
# ----------------------------------------------------------------------------
# Short arm is 2005 (~2,013d): more stable than 2006 (further from the LSTM
# floor) and its mean vol (16.7%) sits between the longer arms, so the contrast
# is cleanly about training length/period, not a stress outlier. 2008 dropped
# (floor check: unstable, seed-std 0.032).
#
# Paste AFTER the setup + floor-check cell has run (all functions defined).
# All three arms end at TRAIN_END (2013-01-01); only the training START moves.
# Validation (calm 2013-14 / stress 2015-16) and test (2020+) are FIXED.
# ============================================================================
import numpy as np

TRAIN_STARTS = {
    "2000_full":  "2000-08-01",   # ~3122d (Paper 1's window)
    "2004_med":   "2004-01-01",   # ~2265d
    "2005_short": "2005-01-01",   # ~2013d (LSTM-stable; replaces unstable 2008)
}

test = sp500_dt[sp500_dt.index >= TEST_START]
actual_full = test["rv_gk"].values

sweep = {}
for variant, tstart in TRAIN_STARTS.items():
    print(f"\n========== TRAINING LENGTH: {variant} (start {tstart}) ==========")
    calm_fc,   n_tr = run_lstm_arm_ts(tstart, CALM_START,   CALM_END,   f"LSTM calm  [{variant}]")
    stress_fc, _    = run_lstm_arm_ts(tstart, STRESS_START, STRESS_END, f"LSTM stress[{variant}]")
    n = min(len(actual_full), len(calm_fc), len(stress_fc))
    actual = actual_full[-n:]; calm_fc = calm_fc[-n:]; stress_fc = stress_fc[-n:]
    for lo, hi in [(33, 67), (25, 75)]:
        qlo, qhi = np.percentile(actual, lo), np.percentile(actual, hi)
        low_mask, high_mask = actual <= qlo, actual >= qhi
        c_hi = qlike_vec(actual, calm_fc)[high_mask].mean()
        s_hi = qlike_vec(actual, stress_fc)[high_mask].mean()
        c_lo = qlike_vec(actual, calm_fc)[low_mask].mean()
        s_lo = qlike_vec(actual, stress_fc)[low_mask].mean()
        dbar, dm, nm = dm_test(actual, calm_fc, stress_fc, high_mask)
        sweep[(variant, f"{lo}/{hi}")] = dict(
            n_train=n_tr, calm_hi=c_hi, stress_hi=s_hi, gap=c_hi-s_hi, dm=dm, n_high=nm,
            calm_lo=c_lo, stress_lo=s_lo)
        print(f"  [{lo}/{hi}] HIGH: calm={c_hi:.3f} stress={s_hi:.3f} gap={c_hi-s_hi:+.3f} "
              f"DM={dm:.2f} n_high={nm} | LOW: calm={c_lo:.3f} stress={s_lo:.3f}")

print("\n\n========= OPTION 2 SUMMARY: gap vs training length, both thresholds =========")
print(f"{'variant':<12}{'thr':>7}{'n_train':>9}{'calm_HV':>10}{'stress_HV':>11}{'gap':>9}{'DM':>8}{'n_high':>8}")
for (variant, thr), r in sweep.items():
    print(f"{variant:<12}{thr:>7}{r['n_train']:>9}{r['calm_hi']:>10.3f}{r['stress_hi']:>11.3f}"
          f"{r['gap']:>+9.3f}{r['dm']:>8.2f}{r['n_high']:>8d}")
print("\nGap positive + DM significant across BOTH thresholds and ALL three lengths")
print("=> validation-regime effect robust to training length (LSTM). Whatever it is, report it.")

In [ ]:
import inspect
# 1. confirm QLIKE selection (not MSE) is the objective
print("val_qlike in train_eval:", inspect.getsource(lstm_train_eval_ts).count("val_qlike"))   # expect >= 2
# 2. confirm leakage guard (history-prepend) is present
print("history-prepend present:", "vstack" in inspect.getsource(lstm_train_eval_ts))           # expect True
# 3. confirm scaler fit on train only
print("train-only scaler:", "fit_transform(tr)" in inspect.getsource(lstm_data_ts))            # expect True
# 4. confirm DM uses corrected HAC lag
print("HAC lag rule:", "2.0/9.0" in inspect.getsource(dm_test))                                 # expect True